# Bronze — Cópia fiel da fonte

Lê todos os arquivos JSON brutos disponíveis no Volume `poc_b3_modernizacao.landing.raw`
(um por dia de execução da ingestão) e grava como tabela Delta na camada Bronze.

Princípio Medallion aplicado aqui: cópia fiel, sem nenhum tratamento de tipo ou conteúdo —
todas as colunas como string, sujeira de origem preservada de propósito. Tipagem, limpeza
e regras de qualidade começam apenas na Silver.

Acumula histórico: cada execução processa todos os arquivos existentes no Volume, não apenas
o do dia — a tabela Bronze cresce conforme novos dias são ingeridos na Landing Zone.

**Entrada:** arquivos JSON em `/Volumes/poc_b3_modernizacao/landing/raw/data=AAAA-MM-DD/`
**Saída:** tabela `poc_b3_modernizacao.bronze.cotacoes`

**Colunas de auditoria incluídas:** `data_carga` (timestamp de ingestão na Bronze),
`arquivo_origem` (rastreabilidade do arquivo fonte), `data_referencia` (data da partição,
extraída do caminho).

In [0]:
%run ../setup/01_utilitarios_pipeline

In [0]:
# observabilidade - marca inicio da execucao
from datetime import datetime
inicio_execucao = datetime.now()

In [0]:
# imports
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType
import json

In [0]:
# widgets - parametros de execucao
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("modo_execucao", "agendado", ["agendado", "reprocessamento_manual"], "Modo de execucao")

In [0]:
# execucao principal - le volume, estrutura, grava bronze, valida, registra observabilidade (com tratamento de erro)
try:
    VOLUME_BASE = "/Volumes/poc_b3_modernizacao/landing/raw"
    MODO_EXECUCAO = dbutils.widgets.get("modo_execucao")

    arquivos_encontrados = []
    for pasta_data in dbutils.fs.ls(VOLUME_BASE):
        if pasta_data.isDir():
            for arquivo in dbutils.fs.ls(pasta_data.path):
                if arquivo.name.endswith(".json"):
                    arquivos_encontrados.append(arquivo.path)

    print(f"Arquivos encontrados: {len(arquivos_encontrados)}")
    for a in arquivos_encontrados:
        print(f"  {a}")

    registros = []
    for caminho_arquivo in arquivos_encontrados:
        data_particao = caminho_arquivo.split("data=")[1].split("/")[0]
        conteudo = json.loads(dbutils.fs.head(caminho_arquivo, 1000000))

        for item in conteudo:
            ticker = item["ticker"]
            resultado = item["body"]["results"][0]
            data_hora_mercado = resultado.get("regularMarketTime", "")
            data_mercado = data_hora_mercado[:10] if data_hora_mercado else None
            data_valida = (data_mercado == data_particao)

            registros.append({
                "ticker": ticker,
                "preco_atual": str(resultado.get("regularMarketPrice", "")),
                "fechamento_anterior": str(resultado.get("regularMarketPreviousClose", "")),
                "data_hora_mercado": data_hora_mercado,
                "data_referencia": data_particao,
                "data_valida": data_valida,
                "arquivo_origem": caminho_arquivo,
            })

    print(f"Total de registros: {len(registros)}")
    for r in registros:
        status = "OK" if r["data_valida"] else "DIVERGENTE"
        print(f"  [{status}] {r['ticker']} - particao={r['data_referencia']}")

    schema = StructType([
        StructField("ticker", StringType(), True),
        StructField("preco_atual", StringType(), True),
        StructField("fechamento_anterior", StringType(), True),
        StructField("data_hora_mercado", StringType(), True),
        StructField("data_referencia", StringType(), True),
        StructField("data_valida", StringType(), True),
        StructField("arquivo_origem", StringType(), True),
    ])

    for r in registros:
        r["data_valida"] = str(r["data_valida"])

    df_bronze = spark.createDataFrame(registros, schema=schema)
    df_bronze = df_bronze.withColumn("data_carga", F.current_timestamp())
    display(df_bronze)

    merge_ou_cria(df_bronze, "poc_b3_modernizacao.bronze.cotacoes", ["ticker", "data_referencia"])

    df_validacao = (spark.table("poc_b3_modernizacao.bronze.cotacoes")
        .select("ticker", "preco_atual", "data_referencia", "data_valida", "data_carga")
        .orderBy("data_referencia", "ticker")
    )
    display(df_validacao)

    data_mais_recente = df_bronze.agg(F.max("data_referencia")).collect()[0][0]

    registrar_execucao(
        notebook="02_bronze",
        data_referencia=data_mais_recente,
        modo_execucao=MODO_EXECUCAO,
        status="sucesso",
        inicio=inicio_execucao,
        fim=datetime.now(),
    )

except Exception as e:
    registrar_execucao(
        notebook="02_bronze",
        data_referencia=None,
        modo_execucao=MODO_EXECUCAO if "MODO_EXECUCAO" in dir() else dbutils.widgets.get("modo_execucao"),
        status="falha",
        inicio=inicio_execucao,
        fim=datetime.now(),
        mensagem_erro=str(e),
    )
    raise